# Previsão de Doenças com Aprendizagem de Máquina
## Projeto Final — Sistemas de Informação | UNIFACISA
**Prof. Bruno Rafael Araújo Vasconcelos**

---

Este notebook implementa um modelo de **Random Forest** para classificar três doenças a partir de sintomas binários:
- `pneumonia`
- `acute bronchitis`
- `cystitis`

## 1. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.feature_selection import SelectFromModel, chi2, SelectKBest
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Configuração visual
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ Bibliotecas importadas com sucesso!')

## 2. Carregamento e Filtragem dos Dados

In [ ]:
# ⚠️ Ajuste o caminho para onde seu arquivo CSV está salvo
CSV_PATH = 'dataset.csv'  # <-- altere se necessário

df_full = pd.read_csv(CSV_PATH)
print(f'Dataset completo: {df_full.shape[0]} linhas, {df_full.shape[1]} colunas')
print(f'Total de doenças únicas: {df_full["diseases"].nunique()}')

In [ ]:
# Filtrando apenas as 3 doenças escolhidas
DISEASES = ['pneumonia', 'acute bronchitis', 'cystitis']

df = df_full[df_full['diseases'].isin(DISEASES)].copy()
df.reset_index(drop=True, inplace=True)

print(f'Dataset filtrado: {df.shape[0]} linhas, {df.shape[1]} colunas')
print('\nDistribuição das classes:')
print(df['diseases'].value_counts())

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
# Distribuição das classes
fig, ax = plt.subplots(figsize=(8, 5))
counts = df['diseases'].value_counts()
bars = ax.bar(counts.index, counts.values, color=sns.color_palette('Set2', 3), edgecolor='white')

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontweight='bold')

ax.set_title('Distribuição das Doenças no Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Doença')
ax.set_ylabel('Quantidade de amostras')
plt.tight_layout()
plt.savefig('distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: distribuicao_classes.png')

In [ ]:
# Verificação de valores ausentes
missing = df.isnull().sum().sum()
print(f'Valores ausentes no dataset: {missing}')
print(f'\nTipos de dados:\n{df.dtypes.value_counts()}')

In [ ]:
# Top sintomas mais frequentes por doença
feature_cols = [c for c in df.columns if c != 'diseases']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = sns.color_palette('Set2', 3)

for ax, disease, color in zip(axes, DISEASES, colors):
    subset = df[df['diseases'] == disease][feature_cols]
    top_symptoms = subset.sum().sort_values(ascending=False).head(10)
    ax.barh(top_symptoms.index[::-1], top_symptoms.values[::-1], color=color, edgecolor='white')
    ax.set_title(f'Top 10 Sintomas\n{disease}', fontweight='bold')
    ax.set_xlabel('Frequência')

plt.suptitle('Sintomas Mais Frequentes por Doença', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('top_sintomas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: top_sintomas.png')

## 4. Preparação dos Dados

In [ ]:
X = df[feature_cols].values
y = df['diseases'].values

# Codificação do target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Classes codificadas:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

print(f'\nShape de X: {X.shape}')
print(f'Shape de y: {y_encoded.shape}')

In [ ]:
# Divisão treino/teste (80/20) com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'Amostras de treino : {X_train.shape[0]}')
print(f'Amostras de teste  : {X_test.shape[0]}')
print(f'Features originais : {X_train.shape[1]}')

## 5. Seleção de Features (Feature Selection)

> Com 377 sintomas binários, muitos podem ser irrelevantes para as 3 doenças escolhidas. 
> A seleção de features reduz overfitting, melhora interpretabilidade e acelera o treinamento.

In [ ]:
# Etapa 1: Remover features com variância zero (sintomas que nunca aparecem ou sempre aparecem)
feature_variance = X_train.var(axis=0)
zero_var_mask = feature_variance > 0

X_train_nzv = X_train[:, zero_var_mask]
X_test_nzv  = X_test[:, zero_var_mask]
feature_names_nzv = np.array(feature_cols)[zero_var_mask]

print(f'Features antes da remoção de variância zero : {X_train.shape[1]}')
print(f'Features após remoção de variância zero     : {X_train_nzv.shape[1]}')
print(f'Features removidas                          : {X_train.shape[1] - X_train_nzv.shape[1]}')

In [ ]:
# Etapa 2: Chi-Quadrado — selecionar as K melhores features estatisticamente relevantes
K_FEATURES = 40  # número de features a manter

selector = SelectKBest(score_func=chi2, k=K_FEATURES)
X_train_sel = selector.fit_transform(X_train_nzv, y_train)
X_test_sel  = selector.transform(X_test_nzv)

selected_mask = selector.get_support()
selected_features = feature_names_nzv[selected_mask]

print(f'Features selecionadas pelo Chi²: {X_train_sel.shape[1]}')
print('\nFeatures selecionadas:')
for f in selected_features:
    print(f'  - {f}')

In [ ]:
# Visualização dos scores Chi² das features selecionadas
scores = selector.scores_[selected_mask]
sorted_idx = np.argsort(scores)[::-1]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(selected_features[sorted_idx][::-1], scores[sorted_idx][::-1],
        color=sns.color_palette('Blues_r', K_FEATURES), edgecolor='white')
ax.set_title(f'Score Chi² das {K_FEATURES} Features Selecionadas', fontsize=14, fontweight='bold')
ax.set_xlabel('Score Chi²')
plt.tight_layout()
plt.savefig('feature_selection_chi2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: feature_selection_chi2.png')

## 6. Treinamento do Modelo — Random Forest

In [ ]:
# Treinamento do Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_sel, y_train)
print('✅ Modelo treinado com sucesso!')
print(f'\nParâmetros do modelo:')
print(f'  - Número de árvores  : {rf_model.n_estimators}')
print(f'  - Features de entrada: {X_train_sel.shape[1]}')
print(f'  - Classes            : {list(le.classes_)}')

## 7. Avaliação do Modelo

In [ ]:
# Predições
y_pred = rf_model.predict(X_test_sel)
y_pred_proba = rf_model.predict_proba(X_test_sel)

# Métricas principais
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall    = recall_score(y_test, y_pred, average='macro')
f1        = f1_score(y_test, y_pred, average='macro')
roc_auc   = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')

print('=' * 45)
print('         MÉTRICAS DO MODELO (TESTE)')
print('=' * 45)
print(f'  Acurácia   : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Precisão   : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print(f'  ROC-AUC    : {roc_auc:.4f}')
print('=' * 45)

In [ ]:
# Relatório completo por classe
print('Relatório de Classificação por Doença:\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Gráfico de barras com as métricas
metrics = {
    'Acurácia' : accuracy,
    'Precisão' : precision,
    'Recall'   : recall,
    'F1-Score' : f1,
    'ROC-AUC'  : roc_auc
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=sns.color_palette('Set2', len(metrics)), edgecolor='white')

for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_title('Métricas de Avaliação — Random Forest', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('metricas_modelo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: metricas_modelo.png')

## 8. Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap='Blues')

ax.set_title('Matriz de Confusão — Random Forest', fontsize=14, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: matriz_confusao.png')

# Análise da matriz
print('\nAnálise da Matriz de Confusão:')
for i, cls in enumerate(le.classes_):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    print(f'  {cls}: TP={tp}, FP={fp}, FN={fn}')

## 9. Importância das Features (Random Forest)

In [ ]:
# Feature importance do Random Forest
importances = rf_model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

top_n = min(20, len(selected_features))
top_features = selected_features[sorted_idx[:top_n]]
top_scores   = importances[sorted_idx[:top_n]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_features[::-1], top_scores[::-1],
        color=sns.color_palette('Greens_r', top_n), edgecolor='white')
ax.set_title(f'Top {top_n} Features Mais Importantes — Random Forest',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importância (Gini)')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: feature_importance_rf.png')

## 10. Validação Cruzada (Cross-Validation)

In [ ]:
# Validação cruzada estratificada com 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(rf_model, X_train_sel, y_train,
                            cv=skf, scoring='accuracy', n_jobs=-1)

print('Validação Cruzada — 5 Folds (dados de treino):')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\n  Média  : {cv_scores.mean():.4f}')
print(f'  Desvio : {cv_scores.std():.4f}')

In [ ]:
# Visualização da validação cruzada
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 6), cv_scores, marker='o', linewidth=2,
        markersize=8, color='steelblue', label='Acurácia por Fold')
ax.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Média: {cv_scores.mean():.4f}')
ax.fill_between(range(1, 6),
                cv_scores.mean() - cv_scores.std(),
                cv_scores.mean() + cv_scores.std(),
                alpha=0.2, color='red', label='±1 Desvio Padrão')
ax.set_xlabel('Fold')
ax.set_ylabel('Acurácia')
ax.set_title('Validação Cruzada — 5 Folds', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: cross_validation.png')

## 11. Comparação: Modelo Completo vs. Modelo com Feature Selection

In [ ]:
# Modelo sem feature selection (todas as features)
rf_full = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=42, n_jobs=-1)
rf_full.fit(X_train, y_train)
y_pred_full = rf_full.predict(X_test)

acc_full = accuracy_score(y_test, y_pred_full)
f1_full  = f1_score(y_test, y_pred_full, average='macro')

# Resumo comparativo
comparison = pd.DataFrame({
    'Modelo'      : ['Todas as Features', f'Feature Selection ({K_FEATURES} feat.)'],
    'Nº Features' : [X_train.shape[1], K_FEATURES],
    'Acurácia'    : [acc_full, accuracy],
    'F1-Score'    : [f1_full, f1]
})

print('Comparação entre modelos:\n')
print(comparison.to_string(index=False))

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_comp = ['#E07B54', '#5B9BD5']

for ax, metric, col in zip(axes, ['Acurácia', 'F1-Score'], ['Acurácia', 'F1-Score']):
    bars = ax.bar(comparison['Modelo'], comparison[metric],
                  color=colors_comp, edgecolor='white', width=0.5)
    for bar, val in zip(bars, comparison[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_title(f'Comparação — {metric}', fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('Impacto da Seleção de Features no Desempenho',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: comparacao_modelos.png')

## 12. Resumo Final e Conclusões

In [ ]:
print('=' * 55)
print('            RESUMO FINAL DO PROJETO')
print('=' * 55)
print(f'  Algoritmo          : Random Forest')
print(f'  Doenças            : {DISEASES}')
print(f'  Amostras totais    : {len(df)}')
print(f'  Features originais : {len(feature_cols)}')
print(f'  Features selecion. : {K_FEATURES}')
print(f'  Split Treino/Teste : 80% / 20%')
print('-----------------------------------------------')
print(f'  Acurácia (teste)   : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Precisão (macro)   : {precision:.4f}')
print(f'  Recall   (macro)   : {recall:.4f}')
print(f'  F1-Score (macro)   : {f1:.4f}')
print(f'  ROC-AUC  (OvR)     : {roc_auc:.4f}')
print(f'  CV Média (5-fold)  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 55)

print("""
CONCLUSÃO:
O modelo Random Forest demonstrou capacidade de distinguir com
alta acurácia as três condições clínicas (pneumonia, bronquite
aguda e cistite) a partir de sintomas binários.

A seleção de features pelo teste Chi² reduziu o espaço de
377 para 40 variáveis sem degradação significativa de
desempenho, reduzindo overfitting e melhorando interpretabilidade.

A validação cruzada estratificada confirmou a estabilidade do
modelo, com baixo desvio padrão entre os folds.
""")